# Anomaly Detection
Statistical anomaly detection for marketing metrics.

Three complementary methods are applied to ad-performance time series:
| Method | Description |
|--------|-------------|
| **Z-score** | Rolling 30-day mean/std; flag if |z| > 3 |
| **IQR** | Flag values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR |
| **% Change** | Flag day-over-day change > 30% |


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.4f}'.format)

from datetime import datetime, timedelta
from config import BASE_DIR, ANOMALY_CONFIG

print("Anomaly detection config:")
for k, v in ANOMALY_CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
from src.extractors import MetaAdsExtractor, GoogleAdsExtractor
from src.transformers import AdsTransformer

# Use a wider date range so the rolling window has enough history
END   = datetime(2025, 3, 1)
START = datetime(2024, 1, 1)

meta_ext = MetaAdsExtractor({"campaigns": 5})
gads_ext = GoogleAdsExtractor({"campaigns": 4})

meta_raw = meta_ext.extract(START, END)
gads_raw = gads_ext.extract(START, END)

ads_t = AdsTransformer()
unified_ads = ads_t.transform(meta_data=meta_raw, google_data=gads_raw)

# Aggregate daily totals for cleaner time-series analysis
daily_spend = (
    unified_ads
    .groupby(['date', 'platform'])
    .agg(spend=('spend', 'sum'), clicks=('clicks', 'sum'),
         conversions=('conversions', 'sum'), impressions=('impressions', 'sum'))
    .reset_index()
)
daily_spend['date'] = pd.to_datetime(daily_spend['date'])
daily_spend = daily_spend.sort_values('date')

print(f"Daily aggregated rows: {len(daily_spend):,}")
print(f"Date range: {daily_spend['date'].min().date()}  →  {daily_spend['date'].max().date()}")
display(daily_spend.head(5))


In [ ]:
from src.quality import AnomalyDetector

detector = AnomalyDetector()

print("Running Z-score anomaly detection on daily spend…")
zscore_anomalies = detector.detect_zscore(
    data           = daily_spend,
    metric_column  = 'spend',
    date_column    = 'date',
    groupby_column = 'platform',
)

print(f"Z-score anomalies detected: {len(zscore_anomalies)}")
if zscore_anomalies:
    za_df = pd.DataFrame(zscore_anomalies)
    display(za_df[['metric', 'date', 'observed_value', 'expected_value',
                    'z_score', 'pct_change', 'severity', 'likely_cause']].head(15))


In [ ]:
# ── Time-series with anomaly points highlighted ───────────────────────────
platforms = daily_spend['platform'].unique()
fig, axes = plt.subplots(len(platforms), 1, figsize=(14, 4 * len(platforms)), sharex=True)
if len(platforms) == 1:
    axes = [axes]

za_df = pd.DataFrame(zscore_anomalies) if zscore_anomalies else pd.DataFrame()

for ax, platform in zip(axes, platforms):
    subset = daily_spend[daily_spend['platform'] == platform].copy()
    ax.plot(subset['date'], subset['spend'], color='#1976D2', linewidth=1, label='Daily Spend')
    ax.fill_between(subset['date'], subset['spend'], alpha=0.15, color='#1976D2')

    # Overlay anomalies
    if not za_df.empty and 'date' in za_df.columns:
        anom_dates_critical = [
            pd.to_datetime(r['date']) for r in zscore_anomalies
            if platform in r.get('metric', '') and r.get('severity') == 'CRITICAL'
        ]
        anom_dates_warning = [
            pd.to_datetime(r['date']) for r in zscore_anomalies
            if platform in r.get('metric', '') and r.get('severity') == 'WARNING'
        ]
        for d in anom_dates_critical:
            val = subset.loc[subset['date'] == d, 'spend']
            if not val.empty:
                ax.scatter([d], val.values, color='red', zorder=5, s=60, marker='v', label='Critical anomaly')
        for d in anom_dates_warning:
            val = subset.loc[subset['date'] == d, 'spend']
            if not val.empty:
                ax.scatter([d], val.values, color='orange', zorder=5, s=40, marker='^', label='Warning anomaly')

    ax.set_title(f"{platform} – Daily Ad Spend with Anomalies", fontweight='bold')
    ax.set_ylabel("Spend ($)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

    # Deduplicate legend
    handles, labels_ = ax.get_legend_handles_labels()
    seen = {}
    for h, l in zip(handles, labels_):
        seen[l] = h
    ax.legend(seen.values(), seen.keys(), loc='upper left', fontsize=8)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(BASE_DIR / "visuals" / "nb05_anomalies_timeseries.png", dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved.")


In [ ]:
# ── IQR method comparison ─────────────────────────────────────────────────
print("Running IQR anomaly detection on daily spend…")
iqr_anomalies = detector.detect_iqr(
    data           = daily_spend,
    metric_column  = 'spend',
    date_column    = 'date',
    groupby_column = 'platform',
)
print(f"IQR anomalies detected: {len(iqr_anomalies)}")

# Method comparison
comparison = pd.DataFrame([
    {"Method": "Z-score (rolling 30d)", "Anomalies Found": len(zscore_anomalies),
     "Notes": f"Threshold: |z| > {ANOMALY_CONFIG['z_score_threshold']}"},
    {"Method": "IQR",                   "Anomalies Found": len(iqr_anomalies),
     "Notes": f"Multiplier: {ANOMALY_CONFIG['iqr_multiplier']}"},
])
print()
print("=== Method Comparison ===")
display(comparison.set_index("Method"))

if iqr_anomalies:
    iqr_df = pd.DataFrame(iqr_anomalies)
    print()
    print("IQR anomaly sample:")
    display_cols = [c for c in ['metric', 'date', 'observed_value', 'expected_range',
                                  'iqr', 'severity'] if c in iqr_df.columns]
    display(iqr_df[display_cols].head(10))


In [ ]:
# ── Combined anomaly alert output ─────────────────────────────────────────
all_anomalies = detector.detect_all(
    data           = daily_spend,
    metrics        = ['spend', 'clicks', 'conversions'],
    date_column    = 'date',
    groupby_column = 'platform',
)

all_df = pd.DataFrame(all_anomalies)
print(f"=== Combined Anomaly Alert Output ===")
print(f"Total unique anomalies across all methods and metrics: {len(all_df)}")

if not all_df.empty:
    critical = all_df[all_df['severity'] == 'CRITICAL']
    warnings_a = all_df[all_df['severity'] == 'WARNING']
    print(f"  Critical: {len(critical)}")
    print(f"  Warning : {len(warnings_a)}")

    print()
    print("Breakdown by method:")
    display(all_df['method'].value_counts().rename("count").to_frame())

    print()
    print("Most severe anomalies:")
    show_cols = [c for c in ['method', 'metric', 'date', 'observed_value',
                               'severity', 'likely_cause', 'recommendation'] if c in all_df.columns]
    display(all_df[all_df['severity'] == 'CRITICAL'][show_cols].head(10))
else:
    print("No anomalies found across all methods – data is within normal bounds.")
